# PEP8_OFRS_Home_Safety_ML_model.
## Home Safety Risk Ranking


This notebook guides you through building a **risk-ranking model** for the home-safety dataset.

The goal is **not** to create a perfect yes/no classifier. The goal is to sort records from highest risk to lowest risk so that the top records contain more true incidents than we would get by random selection.

This is important because the target is imbalanced. When positives are rare, accuracy and a default 0.5 threshold can be misleading. Here we care about questions like:

- If we inspect the **top 50** highest-risk records, how many true incidents do we capture?
- Is the top 50 list better than random selection?
- Would top 100 or top 200 be more realistic operationally?

This simplified version focuses on **one model family: Random Forest**.

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-danger">

## Overall Assessment

The modelling approach is generally well structured and demonstrates genuine predictive     
value for identifying higher-risk properties. The Random Forest model handles the highly      
imbalanced target appropriately through the use of class weighting and achieves    
promising test performance, particularly when used as a ranking and prioritisation     
tool rather than a binary classifier.    

The strongest operational results are observed within the highest-risk records, where     
the model captures a substantial proportion of incidents while maintaining meaningful     
lift above random selection.      
  
However, I have a number of concerns regarding feature engineering and preprocessing.    
Several variables appear to represent coded categorical or ordinal classifications but      
are currently treated as continuous numeric features based solely on their pandas data     
type. In addition, some preprocessing steps, including standardisation of binary     
indicator variables, may be unnecessary for a Random Forest model.     
      
I would recommend reviewing feature classification and preprocessing before relying on      
the tuned model as a final production-ready solution.     
   
---

## Key Strengths

### Appropriate Handling of Class Imbalance

- The target is highly imbalanced (~0.6% positive incidents).     
- `class_weight="balanced_subsample"` has been used appropriately.       
- Evaluation focuses on ROC-AUC, Average Precision and top-k capture rather than accuracy alone.     

### Strong Risk Ranking Performance

The final tuned Random Forest achieved the following performance on the held-out test data:      

| Metric | Test Score |
|----------|----------:|
| ROC-AUC | 0.830 |
| Average Precision | 0.386 |

These results indicate that the model successfully ranks "incident" cases above "non-incident" 
cases despite the extremely imbalanced target.     
   
### Effective Top-K Prioritisation
      
The model performs best as a prioritisation tool.    

| Top K Records | Incidents Captured | Recall |
|--------------|-------------------:|--------:|
| 50 | 7 | 46.7% |
| 100 | 9 | 60.0% |
| 500 | 10 | 66.7% |
    
The top 50 ranked records capture almost half of all incidents while producing     
approximately 23x lift over random selection.      
    
### Robust Hyperparameter Tuning
    
- A baseline Random Forest was established.    
- Hyperparameters were tuned using `RandomizedSearchCV`.     
- Cross-validation was stratified.    
- Average Precision was used as the optimisation metric.     
   
---

## Major Review Comments

### Feature Classification Appears Dtype-Driven

Feature assignment is currently determined by pandas dtypes rather than the business    
meaning of the variables.      

Several fields currently treated as numeric appear to represent coded categories,      
bands or ordinal classifications rather than continuous numeric measurements.      
    
Examples include:    
- `(H) Age - Fine`
- `(H) Household Composition`
- `(H) Family Lifestage v3`
- `(H) Tenure 2011`
- `(H) Household Income v3 - Bands`
- `(H) Affluence v2`

This results in these variables being passed through the numeric preprocessing      
pipeline and standardised by `StandardScaler()` despite potentially being more       
appropriately treated as categorical or ordinal features.      

I recommend reviewing feature classification based on business meaning rather than    
storage dtype before relying on the final model results.     

<div class="alert alert-block alert-danger">

## 1. Imports

Run this cell first. It loads the packages needed for the workflow.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.ensemble
import seaborn as sns
from IPython.display import display
from scipy.stats import randint
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pathlib import Path
#run the functions notebook which contains all the custome ASC data prep functions

%run include_functions.ipynb
%run evaluation_functions.ipynb
warnings.filterwarnings("ignore")

## 2. Simple configuration

Update only the values in this cell.

The most important settings are:

- `DATA_PATH`: where the dataset is saved.
- `TARGET_RAW`: the original target column in the dataset.
- `TOP_K_VALUES`: the ranked list sizes to evaluate.

In [ ]:
# Path to match where your dataset is saved.
df_raw = pd.read_excel("OXON_HH_DATA.xlsx")

# Confirm this is the correct target column in your dataset.
TARGET_RAW = "Incident?"

# This will be the cleaned 0/1 target column used for modelling.
TARGET = "target_incident"

# These are the ranked list sizes we want to evaluate.
# Keep this simple at first.
TOP_K_VALUES = [50, 100, 200, 500]

# Random seed so results are reproducible.
RANDOM_STATE = 42

# Data split sizes.
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN = 0.25

# Random search size.
# Start with 25. Increase later if the notebook runs quickly.
# on my lap top the fitting process takes an exceedingly long time
# ... feel free to amend as per processing time of your system
N_ITER_SEARCH = 10

# Cross-validation folds.## 5 originally
CV_SPLITS = 5

# Turn off code that I do not want to run 
TURN_ON = False

In [ ]:
df_raw.head(3)

In [ ]:
#Station_Ground_Code='JX06'
#df_raw = df_raw[df_raw['Station_Ground_Code']==Station_Ground_Code]

# The region of Oxford City includes two Station Grounds, therefore use the following OR condition:
#df_raw = df_raw[(df_raw['Station_Ground_Code']== 'JX21') | (df_raw['Station_Ground_Code'] == "JX30")]
df_raw = df_raw.sample(100000)

df_raw.shape
#df_raw[['Station_Ground_Code','Incident?']].value_counts()

## Target variable balance?

In [ ]:
df_raw["Incident?"].value_counts(dropna=False)

In [ ]:
df_raw["Incident?"].value_counts(dropna=False, normalize  =True)

<div class="alert alert-block alert-danger">
### The target value is highly imbalanced

<div class="alert alert-block alert-danger
    

<h1 style="text-align: center">Investigating the missing values in features</h1>

<div class="alert alert-block alert-danger">

## Import libraries and custom functions

## Investigate the raw data:

In [ ]:
df_raw.info()

In [ ]:
## check nulls present in raw dataset
null_count(df_raw)

In [ ]:
nulls = null_count(df_raw)
null_list = list(nulls[nulls['Null Count']!=0]['Field'])
null_list

In [ ]:
# Check out the missing values
missing_columns = null_list
#    [    
    #"(H) Household Composition",
    #"(H) Family Lifestage v3"
#]

df_raw[null_list].isna().sum()

In [ ]:
# check distribution of features with missing values:


In [ ]:
fig, ((ax0, ax1)) = plt.subplots(nrows=1, ncols=2)

#colors = ['red', 'tan', 'lime']
ax0.hist(df_raw['(H) Household Composition'],edgecolor = "black")
ax0.set_title('(H) Household Composition',)
ax0.set_xlabel('Classification')
ax0.set_ylabel('Freq')


ax1.hist(df_raw['(H) Family Lifestage v3'],edgecolor = "black")
ax1.set_title('(H) Family Lifestage v3')
ax1.set_ylabel('Freq')
ax1.set_xlabel('Classification')

fig.tight_layout()
plt.show()

In [ ]:
df_raw.assign(
    household_missing_data = (
        df_raw["(H) Family Lifestage v3"].isna()
    ),
    incident_flag=(
        df_raw["Incident?"].map(
            {
                "N": 0,
                "Y": 1,
            }
        )
    ),
).groupby(
    "household_missing_data"
)["incident_flag"].agg(
    ["count", "sum", "mean"]
)

In [ ]:
df_raw.assign(
    household_missing_data = (
        df_raw["(H) Household Composition"].isna()
    ),
    incident_flag=(
        df_raw["Incident?"].map(
            {
                "N": 0,
                "Y": 1,
            }
        )
    ),
).groupby(
    "household_missing_data"
)["incident_flag"].agg(
    ["count", "sum", "mean"]
)

<div class="alert alert-block alert-warning">

## Review comment:

The two household-level fields with missing values have identical missingness patterns,      
with 1,616 records missing both fields.    
</p>    

<p>
I checked whether these records have a materially different incident rate from the rest     
of the dataset.
</p>

<p>
The incident rate is <b>0.80%</b> where household data is missing compared with      
<b>0.94%</b> where it is present, so missingness does not appear to be strongly       
associated with the target.      
</p>

<p>
The current most-frequent imputation approach is therefore reasonable, although       
treating missing values as a separate category could also be considered for       
transparency.        
</p>

</div>

<div class="alert alert-block alert-danger">

### Check Distributions

In [ ]:
df_summary = df_raw.copy()
df_summary = df_summary.drop('s' , axis=1)
df_summary.head(2)

In [ ]:
df_summary_list = df_summary[list(df_summary.select_dtypes(['int64']).columns)]

In [ ]:
numeric_features = df_summary_list[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]].columns#.tolist()



ordinal_features = df_summary_list[['(H) Age - Fine',
                            '(H) Length of Residency',
                            '(H) Affluence v2',
                            '(H) Household Income v3 - Bands']].columns

In [ ]:
ax_1 = df_summary[numeric_features].plot.box(vert=False)
ax_2 = df_summary[ordinal_features].plot.box(vert=False)

In [ ]:
df_raw.shape

<div class="alert alert-block alert-danger">


<h1 style="text-align: center">drop columns that are no longer required, in list form</h1>

<div class="alert alert-block alert-danger">

In [ ]:
# drop columns that are no longer required, in list form
# These are columns which duplicate the information, such as key codes from the SQL database.
columns_to_drop = [
    "ABP_Classification_Code",
    "(H) Mosaic UK 7 Type",
    "(H) Property Type 2011"
]

df_raw.drop(
    columns=columns_to_drop,
    inplace=True,
    errors="ignore"
)

# rename columns 
columns = {
    "s": "Addressbase UPRN",
    "irs.Incident?": "Incident?"
}
df_raw.rename(
    columns=columns,
    inplace=True
)

<div class="alert alert-block alert-danger">

In [ ]:
df_raw.shape

In [ ]:
df_raw.info()

Count the frequency of each unique value in a list (e.g., a DataFrame column) and return the result 
as a mini table including a percentage column (pandas DataFrame)

In [ ]:
# note:  function labelled, 'count_column_values()' is derived from the include_functions.ipynb
count_column_values(df_raw['(H) Fuel Poverty v2 Flag'])

In [ ]:
count_column_values(df_raw['(H) Presence of Elderly Parent'])

In [ ]:
count_column_values(df_raw['(H) Water Poverty Flag'])

In [ ]:
count_column_values(df_raw['Communal Household Flag'])

In [ ]:
# List the columns you wnat to check
columns_to_check = [
    '(H) Fuel Poverty v2 Flag',
    '(H) Presence of Elderly Parent',
    '(H) Water Poverty Flag',
    'Communal Household Flag']

# The reuse the variable in a shorted code (and reuse it later)
df_raw[columns_to_check].info()

<div class="alert alert-block alert-danger">

<h1 style="text-align: center">An alternative approach to Boolean</h1>

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-warning">

## Review comment:

Several binary indicator variables are converted to Boolean values prior to modelling.    
For a Random Forest classifier there is generally little advantage in representing     
binary features as Boolean rather than integer 0/1 values, as the model can naturally      
handle either representation.
     
Furthermore, two of the fields are already stored as 0/1 integers while the remaining    
two are Y/N strings. A simpler and more consistent approach may be to map the Y/N      
fields directly to 0/1 integers and retain all four features in the same format.       

</div>

In [ ]:
binary_columns=[
    "(H) Fuel Poverty v2 Flag",
    "Communal Household Flag"
]

for column in binary_columns:
    df_raw[column] = df_raw[column].map(
        {
            "N": 0, 
            "Y": 1
        }
    )
df_raw.head(3)

In [ ]:
# Now we can check the column again 
df_raw[columns_to_check].info()

In [ ]:
df_raw['(H) Fuel Poverty v2 Flag'].value_counts()

In [ ]:
df_raw['Communal Household Flag'].value_counts()

<div class="alert alert-block alert-danger">

<h1 style="text-align: center">Do the imbalanced features carry any signal?</h1>

<div class="alert alert-block alert-danger">

In [ ]:
pd.crosstab(
    df_raw["(H) Fuel Poverty v2 Flag"],
    df_raw["Incident?"],
    normalize="index"
)

<div class="alert alert-block alert-warning">

## Review comment:

The feature <code>(H) Fuel Poverty v2 Flag</code> is highly imbalanced, with only     
around 0.3% of records in the positive category.<br><br>        

However, the main concern is not the imbalance itself, but the limited difference in     
incident rates between the two groups. Records where the flag is 0 have an incident       
rate of <b>0.84%</b>, compared with <b>2.00%</b> where the flag is 1.<br><br>      
        
This suggests that the feature may contain only a weak signal for the target. It should      
not necessarily be removed at this stage, but its usefulness should be assessed through      
model-based feature importance, permutation importance, or comparison of model      
performance with and without the feature.       
      
</div>

In [ ]:
pd.crosstab(
    df_raw["(H) Water Poverty Flag"],
    df_raw["Incident?"],
    normalize="index"
)

<div class="alert alert-block alert-warning">

## Review comment:

The feature <code>(H) Water Poverty Flag</code> is highly imbalanced, with only     
around 0.3% of records in the positive category.<br><br>        

However, the main concern is not the imbalance itself, but the limited difference in     
incident rates between the two groups. Records where the flag is 0 have an incident       
rate of <b>0.81%</b>, compared with <b>2.04%</b> where the flag is 1.<br><br>      
        
This suggests that the feature may contain only a weak signal for the target. It should      
not necessarily be removed at this stage, but its usefulness should be assessed through      
model-based feature importance, permutation importance, or comparison of model      
performance with and without the feature.       
      
</div>

In [ ]:
pd.crosstab(
    df_raw["Communal Household Flag"],
    df_raw["Incident?"],
    normalize="index"
)

<div class="alert alert-block alert-warning">
      
## Review comment:
       
Although <code>Communal Household Flag</code> is highly imbalanced, with only around      
0.8% of records in the positive category, it appears to contain a strong signal for the        
target.<br><br>      
      
The incident rate is <b>14.30%</b> where <code>Communal Household Flag = 1</code>,      
compared with only <b>0.84%</b> where <code>Communal Household Flag = 0</code>.<br><br>      
      
This suggests the feature should not be removed purely because it is imbalanced.     
For a Random Forest classifier, an imbalanced binary predictor can still be useful if      
the minority group has a materially different target rate, as appears to be the case      
here.       
      
</div>

In [ ]:
pd.crosstab(
    df_raw["(H) Presence of Elderly Parent"],
    df_raw["Incident?"],
    normalize="index"
)

<div class="alert alert-block alert-warning">

## Review comment:

The feature <code>(H) Presence of Elderly Parent</code> is highly imbalanced, with only     
around 0.5% of records in the positive category.<br><br>        

However, the main concern is not the imbalance itself, but the limited difference in     
incident rates between the two groups. Records where the flag is 0 have an incident       
rate of <b>0.94%</b>, compared with <b>1.11%</b> where the flag is 1.<br><br>      
        
This suggests that the feature may contain only a weak signal for the target. It should      
not necessarily be removed at this stage, but its usefulness should be assessed through      
model-based feature importance, permutation importance, or comparison of model      
performance with and without the feature.       
      
</div>

<div class="alert alert-block alert-danger">

In [ ]:
df_raw["Incident?"].value_counts(dropna=False)

<div class="alert alert-block alert-danger">

<h1 style="text-align: center">An alternative approach to consider for cleaning TARGET</h1>

<div class="alert alert-block alert-danger">

**Observations**
1) The target EDA demonstrates that Incident? contains only "Y" and "N" values.      
2) The clean_binary_target() function appears considerably more generic than required for this dataset and      
includes several mappings that are not observed in the source data (yes, true, incident, fire, etc.).      
          
3) In the code block values are converted to lowercase before mapping, the entries "Y" and "N" in the positive/negative sets        
can never be matched and may be removed.        

4) Consider replacing the function with a simple explicit mapping from "Y" → 1 and "N" → 0,      
or adding a comment explaining why a more generic mapping is required.         

In [ ]:
# Build a new modelling dataframe
df = df_raw.copy()

# convert target from Y/N to 1/0
# use the map function to map Y to 1 and N to 0
df[TARGET] = (
    df[TARGET_RAW]
    .str.strip()                  # remove any possible whitespace
    .str.upper()                  # convert all text to uppercase -safey code 
    .map(
        {
            "N": 0,
            "Y": 1
        }
    )
    .astype("Int64")
)

# Validate tat the mapping has wprked as expected 
# If he column contains anything unexpeted you will get an error message 
if df[TARGET].isna().any():
    raise ValueError(
        f"Unexpected values found in "
        f" '{TARGET_RAW}' ."
    )

# visually check the values in TARGET
print("\nCleaned target values:")
display(
    df[TARGET]
    .value_counts(dropna=False)
    .to_frame("count")
)


<div class="alert alert-block alert-danger">

## 5. Check class imbalance

This tells us how rare the positive class is.

If the positive class is rare, a model can have high accuracy while still being useless. That is why this notebook focuses on ranking metrics instead.

In [ ]:
target_summary = df[TARGET].value_counts().sort_index().to_frame("count")
target_summary["proportion"] = target_summary["count"] / target_summary["count"].sum()
display(target_summary)

positive_rate = df[TARGET].mean()
print(f"Raw data positive rate: {positive_rate:.2%}")
print("This is the approximate success rate expected from random selection.")

## Feature Engineering
#### from feature Engineering.ipynb

In [ ]:
df.hist(bins = 30,figsize = (20,15));
plt.show()

# CORRELATIONS

Step 2a: Feature Grouping by data type and preprocessing tasks

In [ ]:
### added 3/8/26:
# Explicitly define the pipeline features rather than rely on pandas data identification.

numeric_features = df[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]].columns#.tolist()

categorical_features = df[['ABP_Classification_Desc',
                                '(H) Mosaic UK 7 Type Label',
                                'Station_Ground_Code',
                                '(H) Tenure 2011',
                                '(H) Fuel Poverty v2 Flag',
                                '(H) Presence of Elderly Parent',
                                '(H) Water Poverty Flag',
                                'Communal Household Flag',
                                # added 5/8/26
                                '(H) Household Composition', 
                                '(H) Family Lifestage v3',
                                    ]].columns#.tolist()

ordinal_features = df[['(H) Age - Fine',
                            '(H) Length of Residency',
                            '(H) Affluence v2',
                            '(H) Household Income v3 - Bands'
                                ]].columns#.tolist()
    
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")

### This function removes columns for which their correlation is greater than a threshold

In [ ]:
from scipy.stats import pearsonr

def remove_correlated(df, cor_thresh1):
    
    """
    Remove DataFrame columns with correlation to target column.
    Correlataion threshold being set as the arguement 'cor_thresh1'
    """
    from copy import deepcopy 

    df_minimal = deepcopy(df) # Needed to not change the original.
    names_df = df.columns.values

    for i in names_df:
        names_df = df_minimal.columns.values
        if i not in names_df:
            continue
            
        i1 = np.where(names_df == i)[0][0]
        
        for j in names_df:
            names_df = df_minimal.columns.values
            j1 = np.where(names_df == j)[0][0]
            
            if j1> i1:
                cor_r,_ = pearsonr(df_minimal.iloc[:,i1],df_minimal.iloc[:,j1])
                if abs(cor_r) > abs(cor_thresh1):
                    df_minimal.drop(names_df[j1], axis=1, inplace=True) 

    return df_minimal

## NUMERIC FEATURES

In [ ]:
import pandas as pd

# Sample DataFrame with NUMERIC features
data_num = df[numeric_features]

df_num = pd.DataFrame(data_num)

# Calculate Pearson correlation matrix
pearson_corr = df_num.corr(method='pearson')

pearson_corr

In [ ]:
cor_target = abs(pearson_corr)
relevant_features = cor_target[pearson_corr<0.5]
#df[relevant_features.index].corr(method='kendall')
relevant_features

#### Numeric features reduction by means of Correlation

In [ ]:
df_num.head(2)

In [ ]:
cor_thresh1 = 0.5
df_num_min = remove_correlated(df_num, cor_thresh1)
df_num_min.head(2)

In [ ]:
if df_num.columns.any() == df_num_min.columns.any():
    print("no columns dropped")
else:
    print(f"columns dropped?:{df_num.columns != df_num_min.columns}")

## ORDINAL FEATURES

In [ ]:
# Sample DataFrame with ORDINAL features
data_ord = df[ordinal_features]

df_ord = pd.DataFrame(data_ord)

# Calculate Pearson correlation matrix
kendall_corr = df_ord.corr(method='kendall')

kendall_corr

In [ ]:
cor_target = abs(kendall_corr)
relevant_features = cor_target[kendall_corr<0.5]
#df[relevant_features.index].corr(method='kendall')
relevant_features

In [ ]:
import seaborn as sns


#corr_pearson = ordinals.select_dtypes('number').corr()
corr_pearson = df_ord.corr()
# plot the heatmap
sns.heatmap(corr_pearson, cmap="Blues",annot = True)
plt.title('Correlation: Ordinals')

In [ ]:
cor_thresh1 = 0.5
df_ord_min = remove_correlated(df_ord, cor_thresh1)
df_ord_min#.head(2)

In [ ]:
if df_ord.columns.any() == df_ord_min.columns.any():
    print("no columns dropped")
else:
    print(f"columns dropped?:{df_ord.columns != df_ord_min.columns}")


For the .corr() function to operate, the target feature must be converted to numeric format:
Replace the TARGET varable from Y/N to 0/1

## For NUMERICAL data

In [ ]:
#corr_spearman = numerics.select_dtypes('number').corr('spearman')
corr_spearman = df_num.select_dtypes('number').corr('spearman')
# plot the heatmap

#corr_pearson = df_ord_min.corr()
sns.heatmap(corr_spearman, cmap="Blues", annot = True)
plt.title('Correlation: Numerical data')

## Step 3:
Update categorical and numeric features lists (because you will possibly be dropping some correlated features)


In [ ]:
numeric_features = df_num_min.columns
ordinal_features = df_ord_min.columns
numeric_features, ordinal_features

## 6. Choose feature columns and avoid leakage

Some columns should not be used as model features.

Common examples:

- unique identifiers;
- post-incident information;
- columns that directly reveal the target;
- columns only known after the incident has happened;
- raw coordinate columns, unless you can justify their use.

The columns removed from modelling can still be kept later in the ranked output so the organisation can identify the records.

In [ ]:

# These columns will NOT be used as model features.
DROP_COLUMNS_AS_FEATURES = [
    TARGET_RAW,
    TARGET,
    "Addressbase UPRN",
    "Unnamed: 0",
    "Easting",
    "Northing",
    "FRSIncidentIdentifier",
    "IncidentCategory",
    "VictimsInvolved",
    "VictimType",
    "WasRescued",
     "LSOA11CD"
]

# These columns are useful for the final ranked output, if they exist.
ID_COLUMNS_FOR_OUTPUT = [
    "Addressbase UPRN",
    "Easting",
    "Northing",
    "LSOA11CD",
    "Property_Type",
    "Property_Description",
]

feature_columns = [
    col for col in df.columns
    if col not in DROP_COLUMNS_AS_FEATURES
]

id_columns = [
    col for col in ID_COLUMNS_FOR_OUTPUT
    if col in df.columns
]

print(f"Number of feature columns: {len(feature_columns)}")
print("\nFeature columns used by the model:")
print(feature_columns)

print("\nID/context columns kept for ranked output:")
print(id_columns)


## End of FEATURE ENGINEERING

# MACHINE LEARNING

## 7. Split the data

We use three sets:

- **Train**: fit the model and tune hyperparameters.
- **Validation**: compare the tuned model and inspect ranking performance.
- **Test**: final honest evaluation, used only once at the end.

Because the target is imbalanced, we use stratified splits so each set has a similar positive rate.

### define the Feature columns

In [ ]:
X = df[feature_columns].copy()
X.info()

## Take a look at the data in X: The features


In [ ]:
X.hist(bins = 30,figsize = (20,15));
plt.show()

In [ ]:
### added 3/8/26:
# explore the 

numerics = X[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]]#.columns#.tolist()

adult_median = numerics['(H) Number of Adults in Household'].median()
child_median = numerics['(H) Number of Children v3'].median()
numerics.hist(bins = 30,figsize = (7,3));

plt.show()
adult_median
child_median


In [ ]:
fig, ((ax0, ax1)) = plt.subplots(nrows=1, ncols=2)

colors = ['red', 'tan', 'lime']
#ax0.hist(numerics['(H) Number of Adults in Household'].median())
ax0.hist(numerics['(H) Number of Adults in Household'],edgecolor = "black")
ax0.legend(prop={'size': 10})
ax0.set_title(f'(H) Number of Adults in Household:\n median_val: {adult_median}')

ax1.hist(numerics['(H) Number of Children v3'],edgecolor = "black")
ax1.legend(prop={'size': 10})
ax1.set_title(f'(H) Number of Children v3:\n median_val: {child_median}')

fig.tight_layout()
plt.show()

In [ ]:

y = df[TARGET].copy()   # TARGET defined part 2. Configuration
ids = df[id_columns].copy() if id_columns else pd.DataFrame(index=df.index)

# First split: train + validation vs test.
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Second split: train vs validation.
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=VALIDATION_SIZE_WITHIN_TRAIN,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "positives": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)



# ---------------------
## FUTURE IMPROVEMENT:
# ---------------------
### If computational Power is not an issue, then the following improvemnets can be made:
## save a regional X_test, y_test into csv for holding.
#### bring this test set in later to compare with full data version

## We shall have two models: one full set and one filter subset
### run evaluation metrics >> choose the best version
#### impossible to view by humand minds, let the model do the work

### Comment
##### The positive rate remains consistent across the train_validation_test splits

## Step 3:
Update categorical and numeric features lists (because you will possibly be dropping some correlated features)

In [ ]:
numeric_features = df_num_min.columns
numeric_features

In [ ]:
ordinal_features = df_ord_min.columns
ordinal_features

In [ ]:
X_numeric_features = X_train[numeric_features]

In [ ]:
X_numeric_features.head(3)

In [ ]:
'''
### added 3/8/26:
# Explicitly define the pipeline features rather than rely on pandas data identification.

numeric_features = X_train[numeric_features].columns

'''
numeric_features = X_train[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]].columns#.tolist()
'''

categorical_features = X_train[['ABP_Classification_Desc',
                                '(H) Mosaic UK 7 Type Label',
                                'Station_Ground_Code',
                                '(H) Tenure 2011',
                                '(H) Fuel Poverty v2 Flag',
                                '(H) Presence of Elderly Parent',
                                '(H) Water Poverty Flag',
                                'Communal Household Flag',
                                # added 5/8/26
                                '(H) Household Composition', 
                                '(H) Family Lifestage v3',
                                    ]].columns#.tolist()

ordinal_features = X_train[ordinal_features].columns

ordinal_features = X_train[['(H) Age - Fine',

                            '(H) Length of Residency',
                            '(H) Affluence v2',
                            '(H) Household Income v3 - Bands'
                                ]].columns#.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")
'''

## 8. Build the preprocessing pipeline

The preprocessing step handles:

- missing numeric values;
- missing categorical values;
- scaling numeric features;
- one-hot encoding categorical features.

Putting preprocessing inside the pipeline helps avoid data leakage during cross-validation.

<div class="alert alert-block alert-warning">

## Review comment:

Feature classification currently appears to be based on pandas dtypes rather than business meaning.      
Several fields stored as integers may represent coded categorical or ordinal classifications rather than true numeric measurements.<br><br>       

For example, <code>(H) Age - Fine</code> contains values 0-11 and appears to represent age bands rather than a continuous age measure.              
Similar concerns may apply to:       
<code>(H) Household Composition</code>,     
<code>(H) Family Lifestage v3</code>,           
<code>(H) Tenure 2011</code>         
<code>(H) Household Income v3 - Bands</code>.<br><br>          

Treating these fields as numeric results in them being passed through the numeric preprocessing pipeline and standardised by <code>StandardScaler</code>.        
It may be more appropriate to explicitly classify features as continuous, ordinal or categorical based on their business definitions rather than their storage dtype.          
      
</div>

## It is important to replace with relevent values.
### As investigated:

### For the numerical data

In [ ]:
num_list = list(numerics)
medians = []
for i in num_list:
    median = numerics[i].median()
    medians.append(median)
    df_medians = pd.DataFrame(medians)

print(f"For any missing Numeric features, replace with their medians:")

df_medians = pd.DataFrame(data=medians, index = num_list)
df_medians

### for the categorical data

In [ ]:
categoricals = X[['ABP_Classification_Desc',
                                '(H) Mosaic UK 7 Type Label',
                                'Station_Ground_Code',
                                '(H) Tenure 2011',
                                '(H) Fuel Poverty v2 Flag',
                                '(H) Presence of Elderly Parent',
                                '(H) Water Poverty Flag',
                                'Communal Household Flag',
                                # added 5/8/26
                                '(H) Household Composition', 
                                '(H) Family Lifestage v3',
                                    ]]
categoricals.hist(bins = 30,figsize = (10,8));
plt.title("Categoric Features Histograms")
plt.show()

In [ ]:
cat_list = list(categoricals)
modes = []
for i in cat_list:
    mode = categoricals[i].mode()
    modes.append(mode)
    modes

print(f"For any missing Categorical features, replace with their most frequent")
df_modes = pd.DataFrame(data=modes, index = cat_list)
df_modes

### for the Ordinal data

In [ ]:
ordinals = X[['(H) Age - Fine',
              '(H) Length of Residency',
            '(H) Affluence v2',
            '(H) Household Income v3 - Bands'
                                ]]
ordinals.hist(bins = 30,figsize = (10,8));
#plt.title("Ordinal Features Histograms")
plt.show()

In [ ]:
ord_list = list(ordinals)
medians = []
for i in ord_list:
    median = ordinals[i].median()
    medians.append(median)
    
    

print(f"For any missing Ordinal features, replace with their medians:")

df_medians = pd.DataFrame(data=medians, index = ord_list)
df_medians

In [ ]:
# The set features as columns references for the pipeline:

numeric_features = X_train[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]].columns#.tolist()


categorical_features = X_train[['ABP_Classification_Desc',
                                '(H) Mosaic UK 7 Type Label',
                                'Station_Ground_Code',
                                '(H) Tenure 2011',
                                '(H) Fuel Poverty v2 Flag',
                                '(H) Presence of Elderly Parent',
                                '(H) Water Poverty Flag',
                                'Communal Household Flag',
                                # added 5/8/26
                                '(H) Household Composition', 
                                '(H) Family Lifestage v3',
                                    ]].columns#.tolist()

ordinal_features = X_train[['(H) Age - Fine',
                            # suppresssed 5/8/26

                            '(H) Length of Residency',
                            '(H) Affluence v2',
                            '(H) Household Income v3 - Bands'
                                ]].columns#.tolist()
    
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")



In [ ]:
### Lgistic Regression Trial:
### benchmark for how Logistic Regression acts

# These columns will NOT be used as model features.
DROP_COLUMNS_AS_FEATURES = [
    TARGET_RAW,
    TARGET,
    "Addressbase UPRN",
    "Unnamed: 0",
    "Easting",
    "Northing",
    "FRSIncidentIdentifier",
    "IncidentCategory",
    "VictimsInvolved",
    "VictimType",
    "WasRescued",
     "LSOA11CD"
]

# These columns are useful for the final ranked output, if they exist.
ID_COLUMNS_FOR_OUTPUT = [
    "Addressbase UPRN",
    "Easting",
    "Northing",
    "LSOA11CD",
    "Property_Type",
    "Property_Description",
]

feature_columns = [
    col for col in df.columns
    if col not in DROP_COLUMNS_AS_FEATURES
]

id_columns = [
    col for col in ID_COLUMNS_FOR_OUTPUT
    if col in df.columns
]

print(f"Number of feature columns: {len(feature_columns)}")
print("\nFeature columns used by the model:")
print(feature_columns)

print("\nID/context columns kept for ranked output:")
print(id_columns)

In [ ]:
y = df[TARGET].copy()
ids = df[id_columns].copy() if id_columns else pd.DataFrame(index=df.index)

# First split: train + validation vs test.
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Second split: train vs validation.
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=VALIDATION_SIZE_WITHIN_TRAIN,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "positives": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)

In [ ]:

'''# The set features as columns references for the pipeline:

numeric_features = X_train[['(H) Number of Adults in Household',
                            '(H) Number of Children v3'
                           ]].columns#.tolist()


categorical_features = X_train[['ABP_Classification_Desc',
                                '(H) Mosaic UK 7 Type Label',
                                'Station_Ground_Code',
                                '(H) Tenure 2011',
                                '(H) Fuel Poverty v2 Flag',
                                '(H) Presence of Elderly Parent',
                                '(H) Water Poverty Flag',
                                'Communal Household Flag',
                                # added 5/8/26
                                '(H) Household Composition', 
                                '(H) Family Lifestage v3',
                                    ]].columns#.tolist()

ordinal_features = X_train[['(H) Age - Fine',
                            # suppresssed 5/8/26

                            '(H) Length of Residency',
                            '(H) Affluence v2',
                            '(H) Household Income v3 - Bands'
                                ]].columns#.tolist()
    
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")
'''

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Compatible with different scikit-learn versions.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot),
])




### Ordinal Encoder
# Added 4/8/26:
ordinal_encoder = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy="median")),
    #("ordinal_encoder", OrdinalEncoder(categories = ordinal_features)),
    ])



preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("ordinal_encoder", ordinal_encoder, ordinal_features )
    ],
    remainder="drop",
)

preprocessor

Primary analytics:

# LOGISTIC REGRESSION:

Train the model (for example logistic regression) using the processed training data.¶

Evaluate its performance using metrics such as accuracy, precision, recall, or ROC-AUC

In [ ]:
## LogisticRegression
from sklearn.linear_model import LogisticRegression
lg_model = LogisticRegression(max_iter = 300)



lg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", sklearn.linear_model.LogisticRegression(   
        penalty='deprecated',
  
    C=1.0,
    l1_ratio=0.0,
    dual=False,
    tol=0.0001,
    fit_intercept=True,
    intercept_scaling=1,
    class_weight=None,
    random_state=None,
    solver='lbfgs',
    max_iter=100,
    verbose=0,
    warm_start=False,
    n_jobs=None,
    )),
])

In [ ]:
## FIT

lg_model.fit(X_train, y_train)

In [ ]:
lg_model_valid_scores = get_scores(lg_model, X_valid)

print("Logistic Regression — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, lg_model_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, lg_model_valid_scores):.3f}")

#display(top_k_capture_table(y_valid, lg_model_valid_scores, TOP_K_VALUES, label="validation"))

In [ ]:
lg_y_test_pred = lg_model.predict(X_train)
lgs = pd.DataFrame(lg_y_test_pred)
lgs.value_counts()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
y_test_pred = lg_model.predict(X_train)
print(f"Accuracy: {accuracy_score(y_train, lg_y_test_pred):.3f}")
print(f"Precision: {precision_score(y_train, lg_y_test_pred):.3f}")
print(f"Recall: {recall_score(y_train, lg_y_test_pred):.3f}")
print(f"RF1 score: {f1_score(y_train, lg_y_test_pred):.3f}")

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Compatible with different scikit-learn versions.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot),
])




### Ordinal Encoder
# Added 4/8/26:
ordinal_encoder = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy="median")),
    #("ordinal_encoder", OrdinalEncoder(categories = ordinal_features)),
    ])



preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("ordinal_encoder", ordinal_encoder, ordinal_features )
    ],
    remainder="drop",
)

preprocessor


## Decision Tree Classifier

In [ ]:
##  DecisionTreeClassifier()

from sklearn.tree import DecisionTreeClassifier



dtc = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", sklearn.tree.DecisionTreeClassifier(   

    )),
])

## FIT
dtc.fit(X_train, y_train)

In [ ]:
dtc_y_test_pred = dtc.predict(X_train)
print(f"Accuracy: {accuracy_score(y_train, dtc_y_test_pred):.3f}")
print(f"Precision: {precision_score(y_train, dtc_y_test_pred):.3f}")
print(f"Recall: {recall_score(y_train, dtc_y_test_pred):.3f}")
print(f"RF1 score: {f1_score(y_train, dtc_y_test_pred):.3f}")

In [ ]:
dtc_predictions = dtc.predict(X_test)
dtc_predictions

In [ ]:
dtc_pred_prob = dtc.predict_proba(X_test)
dtc_pred_prob

In [ ]:
from sklearn.metrics import classification_report

print("classificationReport dtc_predictions")
print(classification_report(y_test, dtc_predictions))


<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-danger">

## 9. Helper functions for ranking evaluation

These functions are provided.

The most important table is the **top-K table**. It answers:

> If we inspect only the top K highest-scored records, how many true positives do we capture?

Important distinction:

- `precision@K` = true positives in top K / K.
- `recall@K` = true positives in top K / all positives in the dataset.

So top-50 recall can look low even when precision is high, because the denominator is all positives, not 50.

In [ ]:
'''
def get_scores(model, X_data):
    """Return the model's positive-class probability scores."""
    return model.predict_proba(X_data)[:, 1]


def top_k_capture_table(y_true, scores, k_values, label="data"):
    """Create a top-K ranking evaluation table."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    n_records = len(y_array)
    total_positives = int(y_array.sum())
    baseline_rate = y_array.mean()

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    rows = []
    for k in k_values:
        k_eff = min(k, n_records)
        positives_in_top_k = int(y_sorted[:k_eff].sum())

        precision_at_k = positives_in_top_k / k_eff if k_eff > 0 else np.nan
        recall_at_k = positives_in_top_k / total_positives if total_positives > 0 else np.nan
        lift_at_k = precision_at_k / baseline_rate if baseline_rate > 0 else np.nan

        # Top K cannot capture more than K positives.
        max_possible_recall = min(k_eff, total_positives) / total_positives if total_positives > 0 else np.nan
        pct_of_max_possible = recall_at_k / max_possible_recall if max_possible_recall > 0 else np.nan

        rows.append({
            "dataset": label,
            "k": k_eff,
            "positives_in_top_k": positives_in_top_k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "max_possible_recall_at_k": max_possible_recall,
            "pct_of_max_possible_recall": pct_of_max_possible,
            "lift_at_k": lift_at_k,
        })

    return pd.DataFrame(rows)


def cumulative_gain_frame(y_true, scores):
    """Return data for a cumulative gains curve."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    cumulative_positives = np.cumsum(y_sorted)
    total_positives = y_sorted.sum()

    return pd.DataFrame({
        "inspected_fraction": np.arange(1, len(y_sorted) + 1) / len(y_sorted),
        "cumulative_capture_rate": cumulative_positives / total_positives if total_positives > 0 else np.nan,
    })
    '''

## 10. Fit a simple <u>baseline</u> Random Forest

This gives a starting point before hyperparameter tuning.

Do not worry if performance is not perfect. We are checking whether the model has useful ranking signal.

In [ ]:
#### The problem being that the algorithm does not like mixed data types such as str and object being in the same list.
#### On 23/6/26 this was fixed by 
#string_cols = X_train.select_dtypes(include=["string","object"]).columns
#X_train[string_cols] = X_train[string_cols].astype("string")
#X_train.info()
#string_cols

In [ ]:
#X_train.info()

## <u>Baseline model</u> = <u>baseline_rf</u>

### RandomForestClassifier : <u>baseline_rf</u>

In [ ]:
## RandomForestClassifier

baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", sklearn.ensemble.RandomForestClassifier(   
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose =0
    )),
])

## FIT
baseline_rf.fit(X_train, y_train)

#baseline_valid_scores = get_scores(baseline_rf, X_valid)

#print("Baseline Random Forest — validation")
#print(f"ROC-AUC:           {roc_auc_score(y_valid, baseline_valid_scores):.3f}")
#print(f"Average Precision: {average_precision_score(y_valid, baseline_valid_scores):.3f}")

#display(top_k_capture_table(y_valid, baseline_valid_scores, TOP_K_VALUES, label="validation"))


In [ ]:
baseline_rf_y_test_pred = baseline_rf.predict(X_train)
print(f"Accuracy: {accuracy_score(y_train, baseline_rf_y_test_pred):.3f}")
print(f"Precision: {precision_score(y_train, baseline_rf_y_test_pred):.3f}")
print(f"Recall: {recall_score(y_train, baseline_rf_y_test_pred):.3f}")
print(f"RF1 score: {f1_score(y_train, baseline_rf_y_test_pred):.3f}")

In [ ]:
baseline_valid_scores = get_scores(baseline_rf, X_valid)

print("Baseline Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, baseline_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, baseline_valid_scores):.3f}")

display(top_k_capture_table(y_valid, baseline_valid_scores, TOP_K_VALUES, label="validation"))

## 11. Tune the Random Forest with RandomizedSearchCV

This searches a wider set of Random Forest settings than a small manual grid.

We tune using **Average Precision** because the target is imbalanced and we care about ranking positives near the top. We still report ROC-AUC because it tells us whether the model has general ranking signal.

Start with `N_ITER_SEARCH = 25`. If the notebook runs quickly, increase it to 50 or 75.
or if your system is slow, reduce N_ITER_SEARCH ~ 10


...Suggestions for improvements:

### >> n_estimators: 
In general the more trees the less likely the algorithm is to overfit. 

So try increasing this. 

The lower this number, the closer the model is to a decision tree, with a restricted feature set.

### >> max_features: 
Try reducing this number (try 30-50% of the number of features). This determines how many features each tree is randomly assigned. 

The smaller, the less likely to overfit, but too small will start to introduce under fitting.


### >> max_depth: 
Experiment with this. This will reduce the complexity of the learned models, lowering over fitting risk. 

Try starting small, say 5-10, and increasing you get the best result.

### >> min_samples_leaf: 
Try setting this to values greater than one. 
This has a similar effect to the max_depth parameter, it means the branch will stop splitting once the leaves have that number of samples each.
...

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", sklearn.ensemble.RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose = 0
    )),
])


'''
-- Output at 125 fits
parameter 125 fits	best_params
model__bootstrap	FALSE
model__class_weight	balanced_subsample
model__max_depth	32
model__max_features	0.3
model__min_samples_leaf	11
model__min_samples_split	25
model__n_estimators	672
'''


param_distributions = {
    "model__bootstrap": [True, False],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "model__max_depth": [ 22,26, 30, 32, 34],
    "model__max_features": ["sqrt", "log2",0.1, 0.2, 0.3,0.4],
    "model__min_samples_leaf": randint(9,14),
    "model__min_samples_split": randint(20,28),
    "model__n_estimators": randint(500, 800),
}


minority_count = int(y_train.value_counts().min())
n_splits = max(2, min(CV_SPLITS, minority_count))

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,
)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
    },
    refit="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
    return_train_score=True,
)


In [ ]:
import time
start_time = time.time()

<div class="alert alert-block alert-danger">

<h1 style="text-align: center">Some notes on the modelling</h1>

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-warning">

## Review comment:

The Random Forest tuning process appears well structured, with appropriate use of      
<code>RandomizedSearchCV</code>, stratified cross-validation, class-weight options and        
Average Precision as the optimisation metric for an imbalanced target.<br><br>       
        
My main concern relates to feature typing rather than model tuning. Several fields       
currently treated as numeric appear to represent coded demographic classifications,        
bands or ordinal categories. Because preprocessing is driven by pandas dtypes, these         
features are routed through the numeric pipeline and standardised before modelling.<br><br>     
      
Before relying on the tuned model results, I would recommend reviewing the business      
meaning of features such as: 
>* <code>(H) Age - Fine</code>
>* <code>(H) Household Composition</code>
>* <code>(H) Family Lifestage v3</code>
>* <code>(H) Tenure 2011</code>
>* <code>(H) Household Income v3 - Bands</code>    

to confirm they are being treated in an appropriate way.      

</div>

<div class="alert alert-block alert-danger">

### The fitting of data through the rf_search may take a significant amount of time.
#### Try reducing the N_ITER_SEARCH to accomodate for any lack of processing power

# FITTING DATA ######################

In [ ]:
rf_search.fit(X_train, y_train)

### <u> OPTIMAL PARAMETERS: </u>

In [ ]:
print(f'N_ITER_SEARCH = {N_ITER_SEARCH}')
print(f'CV_SPLITS = {CV_SPLITS}')
print(f'fits = {N_ITER_SEARCH*CV_SPLITS}')

print("Best CV Average Precision:", rf_search.best_score_)
#print("Best parameters:")
for key, value in rf_search.best_params_.items():
    print(f"  {key}: {value}")


In [ ]:
for key, value in rf_search.best_params_.items():
    df_best_params = pd.DataFrame([{key}, {value}])
 
df_best_params_ = pd.DataFrame(columns = [f'parameter {N_ITER_SEARCH*CV_SPLITS} fits', 'best_params'], data = rf_search.best_params_.items())
df_best_params_


### To assist with researching the optimal parameters per "number of fits", the results may be stored in an Excel workbook
### feel free to amend the location address to suit your system set up
### But no necessary to the overall model

In [ ]:

#with pd.ExcelWriter('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/rf_search_best_params_/best_params.xlsx', engine="openpyxl", mode='a', if_sheet_exists='overlay') as writer:
#    df_best_params_.to_excel(writer, sheet_name=f'fits = {N_ITER_SEARCH*CV_SPLITS}' ,startcol=0,startrow=0)
    #df_best_params_.to_excel(writer, sheet_name=f'fits = {N_ITER_SEARCH*CV_SPLITS}', startcol=1,startrow=1,header=True,index=True)

### Execution time for Optimisation

In [ ]:
print("pipeline_Random_Forest fitted\n--- %s seconds ---" % (time.time() - start_time))

## 12. Evaluate the tuned Random Forest on validation data

This checks whether tuning improved the model on unseen validation data.

Focus especially on:

- ROC-AUC;
- Average Precision;
- positives captured in the top 50, 100, and 200;
- lift compared with random selection;
- percentage of the maximum possible recall@K.

In [ ]:
best_rf = rf_search.best_estimator_
tuned_valid_scores = get_scores(best_rf, X_valid)

print("Tuned Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, tuned_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, tuned_valid_scores):.3f}")

valid_top_k = top_k_capture_table(y_valid, tuned_valid_scores, TOP_K_VALUES, label="validation")
display(valid_top_k)

## 13. Compare baseline and tuned model

Use this section to decide whether tuning actually helped.

A model is not automatically better just because it was tuned. It should improve the validation results or provide a better operational ranking.

In [ ]:
dtc_model_valid_scores = get_scores(dtc, X_valid)
dtc_model_valid_scores

In [ ]:




comparison_rows = []

for model_name, scores in [
    ("Logistic Regression", lg_model_valid_scores),
    ("Decision Tree Classifier", dtc_model_valid_scores),
    ("Baseline RF", baseline_valid_scores),
    ("Tuned RF", tuned_valid_scores),
]:
    top_k = top_k_capture_table(y_valid, scores, TOP_K_VALUES, label="validation")
    top_50_row = top_k[top_k["k"] == min(50, len(y_valid))].iloc[0]

    comparison_rows.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_valid, scores),
        "average_precision": average_precision_score(y_valid, scores),
        "top_50_true_positives": top_50_row["positives_in_top_k"],
        "precision_at_50": top_50_row["precision_at_k"],
        "recall_at_50": top_50_row["recall_at_k"],
        "pct_of_max_possible_recall_at_50": top_50_row["pct_of_max_possible_recall"],
        "lift_at_50": top_50_row["lift_at_k"],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

## 14. Plot the validation cumulative gains curve

This plot shows how quickly the model captures true positives as more records are inspected.

A useful model should rise faster than the diagonal/random line.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

logistic_regression_gain_valid = cumulative_gain_frame(y_valid, lg_model_valid_scores)
baseline_gain_valid = cumulative_gain_frame(y_valid, baseline_valid_scores)
tuned_gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)
'''
ax1 = baseline_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Baseline Random Forest",
)

ax2 = tuned_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Tuned Random Forest",
)
'''

plt.plot(logistic_regression_gain_valid, label = 'Logistic_Reg', color = 'orange')
plt.plot(baseline_gain_valid, label = 'BaseLine_RF', color = 'blue')
plt.plot(tuned_gain_valid,    label = "Tuned_RF", color = 'black')


plt.legend(loc="lower right")

plt.xlabel("Fraction of records inspected")
plt.ylabel("Fraction of positives captured")
plt.title('Cumulative gains curve — validation')
plt.show()

## What is the Cumulative Gains Curve?

AI Overview                 

A cumulative gains curve is a vital evaluation tool used in binary classification to visualize how effectively a predictive model identifies targets compared to random selection. 

It plots the cumulative percentage of true positive targets captured (y-axis) against the cumulative percentage of the population contacted or scored (x-axis).To grasp its business and data science value, understanding a few core elements is key:

How it WorksSorting: The dataset is ordered by the model’s predicted probability of a positive outcome (e.g., customer response or loan default), from highest to lowest.Cumulative Capture: 

The curve plots what percentage of the total actual targets are captured when looking down through this ranked list. For example, the point (10, 30) on the curve means contacting the top 10% of the ranked population catches 30% of all true positive targets.

The Three Key LinesA standard cumulative gains plot contains three primary curves that act as reference points:

The Baseline (Random Model): A straight diagonal line representing random targeting. If you contact X% of the population blindly, you can expect to capture X% of the total targets.

The Predictive Model Curve: The curved line mapping your actual model's performance. It should arch significantly above the baseline towards the top-left corner.

The Optimal Model: The theoretical "perfect" model. This curve shoots straight up to 100% of the targets as quickly mathematically possible (e.g., taking exactly 10% of the population if 10% of the total dataset are positives).

## 15. Final test evaluation = final_model

Only run this after you have finished choosing the model using the validation set.

Do not tune the model again after looking at the test results.

In [ ]:
final_model = best_rf

test_scores = get_scores(final_model, X_test)

print("Final tuned Random Forest — test")
print(f"ROC-AUC:           {roc_auc_score(y_test, test_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_test, test_scores):.3f}")

test_top_k = top_k_capture_table(y_test, test_scores, TOP_K_VALUES, label="test")
display(test_top_k)


<div class="alert alert-block alert-danger">

<h1 style="text-align: center">Some review notes on the modelling</h1>

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-warning">

## Review comment:

The tuned Random Forest shows strong validation performance, with a ROC-AUC of    
<b>0.949</b> and Average Precision of <b>0.423</b>. Given the very low incident     
prevalence, Average Precision and the top-k capture table are more informative than      
accuracy.     

The top-k results suggest the model is effective at ranking high-risk records. The top     
50 records contain <b>8 incidents</b>, capturing approximately <b>53%</b> of validation      
incidents with a precision of <b>16%</b> and a lift of around <b>26x</b>.     
     
However, the validation set appears to contain a small number of positive cases, so the      
top-k results may be sensitive to a small number of records.
      
</div>

<div class="alert alert-block alert-warning">

## Review comment:

The tuned Random Forest shows a drop in performance from validation to test, with    
ROC-AUC reducing from <b>0.949</b> to <b>0.830</b>. Average Precision is more stable,      
falling from <b>0.423</b> to <b>0.386</b>, which is still strong given the very low incident rate.    

The top-k results suggest that the model remains useful as a prioritisation tool on the      
test set. The top 50 records contain <b>7 incidents</b>, capturing approximately      
<b>46.7%</b> of all test incidents with a precision of <b>14%</b>. The top 100 records      
capture <b>9 incidents</b>, or <b>60%</b> of all test incidents.      

However, returns diminish beyond the top 100. The top 200 contains the same number of      
incidents as the top 100, and the top 500 contains only one additional incident. This       
suggests that the model is most useful if applied as a focused high-risk ranking tool        
rather than as a broad screening approach.        
       
Because the test set appears to contain only a small number of positive incidents, these      
top-k results should be interpreted with some caution. A small change in the number of       
captured incidents would materially affect the reported recall.          

</div>

<div class="alert alert-block alert-danger">

## 16. Create a ranked test output

This creates a table sorted from highest predicted risk to lowest predicted risk.

The top rows are the records the organisation would prioritise for inspection or intervention.

In [ ]:
ids_test[:5]

In [ ]:
ranked_test = ids_test.copy()
ranked_test["true_target"] = y_test.values
ranked_test["risk_score"] = test_scores
ranked_test["risk_rank"] = ranked_test["risk_score"].rank(method="first", ascending=False).astype(int)

ranked_test = ranked_test.sort_values("risk_score", ascending=False)
ranked_test = ranked_test.drop_duplicates(subset=['Addressbase UPRN'], keep='first') 
display(ranked_test.head(10))

# Save outputs for reporting.
ranked_test.to_csv("ranked_test_output.csv", index=False)
#ranked_test.head(50).to_csv("top_50_ranked_test_output.csv", index=False)

#print("Saved ranked_test_output.csv")
#print("Saved top_50_ranked_test_output.csv")

In [ ]:
display(ranked_test.head(100))

<div class="alert alert-block alert-danger">

<div class="alert alert-block alert-warning">

## Review comment:

The ranked output contains duplicate <code>Addressbase UPRN</code> values in the highest-risk records.       
For example, the top two ranked records have the same UPRN, LSOA and risk score.      

If the model is intended to produce a ranked list of unique addresses or properties for operational review,       
duplicate records should be investigated and the top-k evaluation should potentially be repeated at unique UPRN level rather than row level.      

If each row represents a distinct event, case or time period, then duplicates may be valid, but this should be clearly documented.       

</div>

<div class="alert alert-block alert-danger">

## 17. How to explain low recall@50

Use this explanation if your top-50 recall looks low.

Recall@50 is calculated as:

```text
true positives in top 50 / all true positives in the dataset
```

So if the top 50 contains 38 true positives, precision is high:

```text
precision@50 = 38 / 50 = 0.76
```

But recall may still look low if there are many positives overall. For example, if there are 200 positives:

```text
recall@50 = 38 / 200 = 0.19
```

That does not mean the model is useless. It means top 50 is too small to capture most positives.

That is why this notebook also reports:

```text
pct_of_max_possible_recall
```

This shows how close the model is to the best possible result for that value of K.

## 18. Interpretation prompts

Answer these in markdown after running the notebook.

1. What is the positive class and why is the problem imbalanced?
2. Why is accuracy not enough for this task?
3. What was the validation ROC-AUC?
4. What was the validation Average Precision?
5. In the validation top 50, how many true positives were captured?
6. What was precision@50?
7. What was recall@50?
8. What percentage of the maximum possible recall@50 did the model achieve?
9. Did tuning improve the baseline Random Forest?
10. Based on the test results, should the organisation use top 50, top 100, or top 200?
11. What extra data might improve the ranking?
12. What are the limitations of using this model operationally?

Suggested conclusion structure:

> The model should be used as a prioritisation tool, not as an automatic decision-maker. The most useful metric is whether the top-ranked records contain more true incidents than random selection. The final top-K results suggest that [...]. However, the model is limited by [...], so future work should [...].

# RESPONSES:

1. What is the positive class and why is the problem imbalanced?
   The positive class is the represented by "1" the binary output of [0,1]
   Which is attributed to the class of an event, in this case an "incident", occuring => 1
   The class attributed to the class of an event, in this case an "incident", NOT  occuring => 0

   Since we are dealing with the whether an incident will occur, output = 1, being extremely rare event within a very large dataset,    the distributions of 0 and 1 are inbalanced. Such as proprtion(0) = 0.99, proportion(1) = 0.01

   ### From Google AI:
   
  Unbalanced (or imbalanced) data in machine learning occurs when one class significantly outnumbers the other(s) in a dataset. For example, in a credit card fraud detection dataset, 99% of transactions might be legitimate (the majority class), while only 1% are fraudulent (the minority class).
Because machine learning algorithms inherently try to maximize overall accuracy, an unbalanced dataset can cause a model to simply guess the majority class every time. This results in high numerical accuracy but terrible performance on the rare, often critical events.

Why It's a ProblemMisleading Metrics: A model predicting 100% legitimate transactions in the example above achieves 99% accuracy, completely failing to catch the 1% fraud.Class Bias: The algorithm never learns the true characteristics of the minority group, leading to high false-negative rates

Work around:
Using the Right Evaluation Metrics: Instead of relying on raw accuracy, you should evaluate model performance using metrics designed for imbalanced data, such as Precision, Recall, F1-Score, or the ROC-AUC curve.
https://www.youtube.com/watch?v=6kwEbsCiLg8

2. Why is accuracy not enough for this task?
Accuracy will suffer from skewing as a result of the raw data being skewed.
If 99% of targets belong to class A, then the model can easily "guess" the prediction as A, by simply saying the target is A will be correct 99/100 guesses
By means of the standard accuracy metric: (TP + TN) / Total === (99 + 0)/100  = 0.99
This masks the models detection of the minority class, such a A = 99, B = 1, it will be difficult to identify class B

Prefered metrics: 
PRECISION: number of pred(+) are actual(+)
RECALL: number of  actual(+) are pred(+)      --- High Recall catches most cases, but may include some FP
High FP - suitable for our task since we are assessing for prevention of incidents, so FP creeping in will have no serious impact/
F1:   denotes balance between PRECISION and RECALL
ROC-AUC

# Join to geospatial data for mapping and standard analysis methods
## Not essential for ML model

In [ ]:
### Extra part.
#### Join geolocation result:
#Gazetter = pd.read_excel('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/OXFORDSHIRE_GAZETTER.xlsx')
#Gazetter = pd.DataFrame(Gazetter)
#Gazetter.drop(columns = ['BUILDINGNAME' , 'BUILDINGNUMBER'])
#Gazetter.head(2)

In [ ]:
#ranked_geolocs = Gazetter.merge(ranked_test, left_on = 'FCL_URN', right_on = 'Addressbase UPRN', how = 'right')
#ranked_geolocs.drop(columns = ['BUILDINGNAME' , 'BUILDINGNUMBER'], inplace = True)
#ranked_geolocs.sample(5) 

In [ ]:
#ranked_geolocs_area = ranked_geolocs[['AREANAME2']].value_counts()
#areas = pd.DataFrame(ranked_geolocs_area)
#areas.to_csv(f'H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_areas{Station_Ground_Code}.csv')
#areas

In [ ]:
#ranked_geolocs.to_csv('H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_geolocs_for_map_.csv')
#ranked_geolocs.to_csv('ranked_geolocs_for_map_.csv')

#streets.to_excel('H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_streets.xlsx')
#streets.to_csv('H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_streets.csv')


#streets.to_excel(f'H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_streets{Station_Ground_Code}.xlsx')
#streets.to_csv(f'H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_streets{Station_Ground_Code}.csv')

# 16. Testing:

In [ ]:
from sklearn.metrics import precision_score, roc_auc_score, mean_squared_error, accuracy_score
from sklearn.metrics import recall_score, classification_report, roc_curve, confusion_matrix
from sklearn import metrics

### TRAINING DATA

In [ ]:
# Model Performance
# we can get performance of the model on the TRAIN data set from 
y_train_pred_best_rf =  best_rf.predict(X_train)
y_train_pred_best_rf

In [ ]:
## 5. Evaluate on TRAINING SET: 
from sklearn.metrics import classification_report
target_names = ['0', '1']
print("TUNED MODEL: \nbest_rf(X_train)\n\nClassification Report:")
print(classification_report(y_train, y_train_pred_best_rf, target_names=target_names))

## TEST DATA

In [ ]:
# 5. Evaluate on TEST SET: 
y_test_pred_best_rf = best_rf.predict(X_test)
y_test_pred_best_rf

target_names = ['0', '1']
print("TUNED MODEL: \nbest_rf(X_test)\n\nClassification Report:")
print(classification_report(y_test, y_test_pred_best_rf, target_names=target_names))

## Accuracy Scores:

In [ ]:
best_rf_pred_RF_ACCURACY =  metrics.accuracy_score( y_train,  y_train_pred_best_rf)
best_rf_pred_RF_PRECISION = metrics.precision_score(y_train,  y_train_pred_best_rf)
best_rf_pred_RF_RECALL =    metrics.recall_score(   y_train,  y_train_pred_best_rf)
best_rf_pred_RF_PF1 =       metrics.f1_score(       y_train,  y_train_pred_best_rf)

In [ ]:
d = {"accuracy_score":  [metrics.accuracy_score(y_train, y_train_pred_best_rf),  metrics.accuracy_score( y_test, y_test_pred_best_rf)], 
     "precision_score": [metrics.precision_score(y_train, y_train_pred_best_rf), metrics.precision_score(y_test, y_test_pred_best_rf)],
     "recall_score":    [metrics.recall_score(y_train, y_train_pred_best_rf),    metrics.recall_score(   y_test, y_test_pred_best_rf)],
     "f1_score":        [metrics.f1_score(y_train, y_train_pred_best_rf),        metrics.f1_score(       y_test, y_test_pred_best_rf)]    
    }
print("RANDOM FOREST baseline: With respect to target variable = 1\n")
RF_metrics = pd.DataFrame(data=d, index=['TRAIN', 'TEST'])
RF_metrics = RF_metrics.T
RF_metrics


In [ ]:
plt.figure(figsize = (10,5))
#RF_metrics[['accuracy_score','precision_score']].plot(kind = 'bar', legend = True)
#RF_metrics[['recall_score',	'f1_score']].plot(kind = 'bar', legend = True)

RF_metrics[['TRAIN','TEST']].plot(kind = 'bar', legend = True)
plt.title("PERFRMANCE METRICS\nRANDOM FOREST baseline model:\nTRAIN : TEST") 
#plt.legend(loc = " right")
plt.legend(loc=(0.8,1.1))
plt.xlabel('Data set')
plt.ylabel('Measure [0,1]')
plt.show()

## ROC metrics

TEST data Prediction Probabilities:

## BASELINE

In [ ]:
y_test_pred_proba_baseline_rf =  baseline_rf.predict_proba(X_test)[:,1]
y_test_pred_proba_baseline_rf

In [ ]:
logit_roc_auc_baseline        = roc_auc_score(y_test,    y_test_pred_proba_baseline_rf)
print(f'ROC-AUC_score[y_test]: {logit_roc_auc_baseline}')

In [ ]:
fpr_baseline_rf, tpr_baseline_rf, thresholds = roc_curve(y_test,    y_test_pred_proba_baseline_rf)

In [ ]:

plt.figure()
plt.plot(fpr_baseline_rf, tpr_baseline_rf, label='BASELINE Random Forest (area = %0.2f)' % logit_roc_auc_baseline,color = 'blue' )
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\n BASELINE_MODEL randomforest \ny_test_pred_proba_baseline_rf')
plt.legend(loc="lower right")
plt.show()


In [ ]:
x = RF_metrics.T['precision_score'][1:][0:] 
#y = RF_metrics['precision_score'][0:][0:]
x


## TUNED ROC:  
#### logit_roc_auc_best_rf

In [ ]:
y_test_pred_proba_best_rf =  best_rf.predict_proba(X_test)[:,1]
y_test_pred_proba_best_rf

In [ ]:
logit_roc_auc_best_rf       = roc_auc_score(y_test,    y_test_pred_proba_best_rf)
print(f'ROC-AUC_score[y_test]: {logit_roc_auc_best_rf}')

In [ ]:
fpr_best_rf, tpr_best_rf, thresholds = roc_curve(y_test,    y_test_pred_proba_best_rf)

In [ ]:
'''
plt.figure()
plt.plot(fpr_best_rf, tpr_best_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_best_rf, color = 'red')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\n TUNED MODEL: random forest\ny_test_pred_proba_best_rf')
plt.legend(loc="lower right")
plt.show()
'''

In [ ]:

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\nRANDOM FOREST \nTUNED_model vs BASELINE_model')


plt.plot(fpr_baseline_rf, tpr_baseline_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_baseline, color = 'blue')
plt.plot(fpr_best_rf, tpr_best_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_best_rf, color = 'red')
plt.plot([0, 1], [0, 1],'r--')

plt.legend(labels = ['baseline_rf','tuned_rf'], loc="lower right")
plt.show()

In [ ]:
data = {

  "Baseline_ROC_Score": logit_roc_auc_baseline,
  "Tuned_RF_ROC_Score": logit_roc_auc_best_rf
}
data

### additional comparison with Logistic Regression & Decision tree models: 

In [ ]:
y_test_pred_proba_lg_model =  lg_model.predict_proba(X_test)[:,1]
y_test_pred_proba_dtc      =  dtc.predict_proba(X_test)[:,1]

In [ ]:
logit_roc_auc_lg_model        = roc_auc_score(y_test,    y_test_pred_proba_lg_model)
logit_roc_auc_dtc             = roc_auc_score(y_test,    y_test_pred_proba_dtc)

print(f'Log_Reg_ROC-AUC_score[y_test]: {logit_roc_auc_lg_model}')
print(f'DTC_ROC-AUC_score[y_test]:     {logit_roc_auc_dtc}')

In [ ]:
fpr_lg_model, tpr_lg_model, thresholds = roc_curve(y_test,    y_test_pred_proba_lg_model)
fpr_dtc,      tpr_dtc,      thresholds = roc_curve(y_test,    y_test_pred_proba_dtc)

In [ ]:

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\nRANDOM FOREST \nTUNED_model vs BASELINE_model')

plt.plot(fpr_lg_model   , tpr_lg_model   , label='Random Forest (area = %0.2f)' %logit_roc_auc_lg_model, color = 'orange')
plt.plot(fpr_dtc        , tpr_dtc        , label='Random Forest (area = %0.2f)' %logit_roc_auc_dtc     , color = 'green')

plt.plot(fpr_baseline_rf, tpr_baseline_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_baseline, color = 'blue')
plt.plot(fpr_best_rf   , tpr_best_rf     , label='Random Forest (area = %0.2f)' % logit_roc_auc_best_rf, color = 'red')
plt.plot([0, 1], [0, 1],'r--')

plt.legend(labels = ['logistic_regression','decision tree classifier', 'baseline_rf','tuned_rf'], loc="lower right")
plt.show()

### feature importance

In [ ]:
best_pipeline=rf_search.best_estimator_ 
print(best_pipeline.named_steps)

In [ ]:
rf_model = best_pipeline.named_steps["model"]
feature_importances = rf_model.feature_importances_
feature_importances.sum()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
rf_model = best_pipeline.named_steps["model"]
feature_importances = rf_model.feature_importances_
preprocessor = best_pipeline.named_steps["preprocessor"]
feature_names = preprocessor.get_feature_names_out()
print("Number of transformed features:", len(feature_names))
print("Number of importances:", len(feature_importances))
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importances
}).sort_values("Importance", ascending=False)
top_features = importance_df.head(20).sort_values("Importance")
plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.title("Top 20 Random Forest Feature Importances")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
 
 

In [ ]:
importance_df.head(20)

### What is a Random Forest?
### It is a collection of Decsion Trees.
### One such Decision Tree can be pulled out at random from the Random Forest Algorithm for display:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

In [ ]:
d_tree = 1

rf_model = best_rf.named_steps["model"]
example_tree = rf_model.estimators_[0]

plt.figure(figsize=(24, 12))
plot_tree(
    example_tree,
    max_depth=2,
    feature_names=feature_names,
    class_names=["NO", "Incident = YES"],
    filled=True,
    impurity=False,
    proportion=True,
    rounded=True,
    fontsize=11
)
plt.title(f"Decision Tree Number: {d_tree}")
plt.tight_layout()
plt.savefig("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/images/DecsionTree_example.svg")#, bbox_inches="tight")
plt.savefig("DecsionTree_example.svg")#, bbox_inches="tight")
plt.show()

In [ ]:
d_tree = 5
rf_model = best_rf.named_steps["model"]
example_tree = rf_model.estimators_[d_tree]

plt.figure(figsize=(24, 12))
plot_tree(
    example_tree,
    max_depth=2,
    feature_names=feature_names,
    class_names=["NO", "Incident = YES"],
    filled=True,
    impurity=False,
    proportion=True,
    rounded=True,
    fontsize=11
)
plt.title(f"Decision Tree Number: {d_tree}")
plt.tight_layout()
plt.savefig("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/images/DecsionTree_example.svg")#, bbox_inches="tight")
plt.savefig("DecsionTree_example.svg")#, bbox_inches="tight")
plt.show()